In [1]:
import os
from pathlib import Path

def count_yolo_dataset(dataset_path):
    dataset_path = Path(dataset_path)
    splits = ["train", "valid", "test"]

    counts = {}
    for split in splits:
        # Look for split folders anywhere inside dataset_path
        img_dirs = list(dataset_path.rglob(f"{split}/images"))
        lbl_dirs = list(dataset_path.rglob(f"{split}/labels"))

        img_count = sum(len(list(d.glob("*.jpg"))) + len(list(d.glob("*.png"))) for d in img_dirs)
        lbl_count = sum(len(list(d.glob("*.txt"))) for d in lbl_dirs)

        # Only include split if any images or labels were found
        if img_count > 0 or lbl_count > 0:
            counts[split] = {
                "images": img_count,
                "labels": lbl_count
            }

    return counts

# Example usage:
if __name__ == "__main__":
    datasets = [
        "my-raccoons.v1i.yolov11",
        "Raccoon.v2-raw.yolov11",
        "Rodent.v2i.yolov11",
        "Rodents.v2i.yolov11",
        "combined_dataset",
        "rebalanced_dataset",
        "augmentation_dataset"
    ]
    for ds in datasets:
        print(f"Dataset: {ds}")
        try:
            counts = count_yolo_dataset(ds)
            if counts:
                for split, stats in counts.items():
                    print(f"  {split}: {stats['images']} images, {stats['labels']} labels")
            else:
                print("  No images/labels found in any split.")
        except Exception as e:
            print(f"  Error reading {ds}: {e}")


Dataset: my-raccoons.v1i.yolov11
  train: 459 images, 459 labels
  valid: 29 images, 29 labels
  test: 17 images, 17 labels
Dataset: Raccoon.v2-raw.yolov11
  train: 150 images, 150 labels
  valid: 29 images, 29 labels
  test: 17 images, 17 labels
Dataset: Rodent.v2i.yolov11
  train: 351 images, 351 labels
  valid: 30 images, 30 labels
  test: 2 images, 2 labels
Dataset: Rodents.v2i.yolov11
  train: 633 images, 633 labels
  valid: 36 images, 36 labels
  test: 27 images, 27 labels
Dataset: combined_dataset
  No images/labels found in any split.
Dataset: rebalanced_dataset
  train: 1930 images, 1930 labels
  valid: 551 images, 551 labels
  test: 277 images, 277 labels
Dataset: augmentation_dataset
  No images/labels found in any split.


In [2]:
import os
import shutil
from pathlib import Path
import uuid
import random

def create_combined_dataset(datasets, output_dir="combined_dataset"):
    """
    Simple function to combine YOLO datasets with unique IDs
    """
    output_path = Path(output_dir)
    
    # Create directories
    for split in ["train", "valid", "test"]:
        (output_path / split / "images").mkdir(parents=True, exist_ok=True)
        (output_path / split / "labels").mkdir(parents=True, exist_ok=True)
    
    # Create mapping record
    mapping = []
    
    for dataset in datasets:
        dataset_path = Path(dataset)
        print(f"Adding: {dataset}")
        
        for split in ["train", "valid", "test"]:
            # Find image and label directories
            img_dir = next(dataset_path.rglob(f"{split}/images"), None)
            lbl_dir = next(dataset_path.rglob(f"{split}/labels"), None)
            
            if img_dir and img_dir.exists():
                # Process all images
                for img_file in img_dir.glob("*.*"):
                    if img_file.suffix.lower() in ['.jpg', '.jpeg', '.png']:
                        # Create unique ID
                        unique_id = str(uuid.uuid4())
                        new_img_name = f"{unique_id}{img_file.suffix}"
                        new_lbl_name = f"{unique_id}.txt"
                        
                        # Copy image
                        shutil.copy2(img_file, output_path / split / "images" / new_img_name)
                        
                        # Copy corresponding label if exists
                        if lbl_dir and lbl_dir.exists():
                            lbl_file = lbl_dir / f"{img_file.stem}.txt"
                            if lbl_file.exists():
                                shutil.copy2(lbl_file, output_path / split / "labels" / new_lbl_name)
                        
                        # Record mapping
                        mapping.append({
                            'dataset': dataset,
                            'original_file': img_file.name,
                            'new_id': unique_id,
                            'split': split
                        })
    
    # Save mapping to CSV
    import pandas as pd
    df = pd.DataFrame(mapping)
    df.to_csv(output_path / "file_mapping.csv", index=False)
    
    print(f"\nCreated combined dataset in: {output_dir}")
    print(f"Total files processed: {len(mapping)}")
    return output_path

# Usage:
datasets = [
    "my-raccoons.v1i.yolov11",
    "Raccoon.v2-raw.yolov11", 
    "Rodent.v2i.yolov11",
    "Rodents.v2i.yolov11"
]

combined_path = create_combined_dataset(datasets)

Adding: my-raccoons.v1i.yolov11
Adding: Raccoon.v2-raw.yolov11
Adding: Rodent.v2i.yolov11
Adding: Rodents.v2i.yolov11

Created combined dataset in: combined_dataset
Total files processed: 1780


In [ ]:
def rebalance_train_test_split(dataset_path, output_dir="rebalanced_dataset", train_ratio=0.70, valid_ratio=0.20, test_ratio=0.10):
    """
    Rebalance a YOLO dataset into a new train/test split (default 70/20/10).
    Works on the already combined dataset without duplicating counts.
    """
    dataset_path = Path(dataset_path)
    output_path = Path(output_dir)

    # Collect all images + labels just once
    all_pairs = []
    for split in ["train", "valid", "test"]:
        img_dir = dataset_path / split / "images"
        lbl_dir = dataset_path / split / "labels"
        if not img_dir.exists():
            continue
        for img_file in img_dir.glob("*.*"):
            if img_file.suffix.lower() in [".jpg", ".jpeg", ".png"]:
                lbl_file = lbl_dir / f"{img_file.stem}.txt"
                if lbl_file.exists():
                    all_pairs.append((img_file, lbl_file))

    print(f"Found {len(all_pairs)} unique image-label pairs in total.")

    # Shuffle
    random.shuffle(all_pairs)

    # Split indices
    n = len(all_pairs)
    train_end = int(n * train_ratio)
    valid_end = train_end + int(n * valid_ratio)

    # Train/Test/Valid split
    train_pairs = all_pairs[:train_end]
    valid_pairs = all_pairs[train_end:valid_end]
    test_pairs  = all_pairs[valid_end:]

    # Reset rebalanced folder
    if output_path.exists():
        shutil.rmtree(output_path)

    for split in ["train", "valid", "test"]:
        (output_path / split / "images").mkdir(parents=True, exist_ok=True)
        (output_path / split / "labels").mkdir(parents=True, exist_ok=True)

    # Copy files into new dataset
    def copy_pairs(pairs, split):
        for img_file, lbl_file in pairs:
            shutil.copy2(img_file, output_path / split / "images" / img_file.name)
            shutil.copy2(lbl_file, output_path / split / "labels" / lbl_file.name)

    copy_pairs(train_pairs, "train")
    copy_pairs(valid_pairs, "valid")
    copy_pairs(test_pairs, "test")

    print(f"Created rebalanced dataset at: {output_path}")
    print(f"{len(train_pairs)} train, {len(valid_pairs)} valid, {len(test_pairs)} test")
    return output_path

rebalanced_path = rebalance_train_test_split(combined_path)

Found 1780 unique image-label pairs in total.
Created rebalanced dataset at: rebalanced_dataset
1246 train, 356 valid, 178 test


In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import numpy as np
from pathlib import Path
import shutil
import uuid
import pandas as pd

class YOLOAugmentation:
    def __init__(self, augmentation_strength='medium'):
        """Initialize augmentation pipeline based on strength"""
        
        if augmentation_strength == 'light':
            self.transform = A.Compose([
                A.HorizontalFlip(p=0.3),
                A.RandomBrightnessContrast(p=0.2),
                A.HueSaturationValue(p=0.2),
                A.RandomGamma(p=0.2),
            ], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))
            
        elif augmentation_strength == 'medium':
            self.transform = A.Compose([
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.2),
                A.RandomRotate90(p=0.3),
                A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
                A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.3),
                A.Blur(blur_limit=3, p=0.2),
                A.MotionBlur(blur_limit=3, p=0.2),
                A.RandomGamma(gamma_limit=(80, 120), p=0.2),
                A.ImageCompression(quality_lower=85, quality_upper=100, p=0.2),
            ], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))
            
        elif augmentation_strength == 'heavy':
            self.transform = A.Compose([
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.3),
                A.RandomRotate90(p=0.5),
                A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.2, rotate_limit=15, p=0.5),
                A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.5),
                A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30, val_shift_limit=20, p=0.4),
                A.Blur(blur_limit=5, p=0.3),
                A.MotionBlur(blur_limit=5, p=0.3),
                A.MedianBlur(blur_limit=3, p=0.2),
                A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
                A.RandomGamma(gamma_limit=(70, 130), p=0.3),
                A.ChannelShuffle(p=0.1),
                A.ImageCompression(quality_lower=75, quality_upper=100, p=0.3),
            ], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))
    
    def read_yolo_label(self, label_path, img_width, img_height):
        """Read YOLO format label file"""
        bboxes = []
        class_labels = []
        
        try:
            with open(label_path, 'r') as f:
                for line in f.readlines():
                    if line.strip():
                        class_id, x_center, y_center, width, height = map(float, line.strip().split())
                        bboxes.append([x_center, y_center, width, height])
                        class_labels.append(class_id)
        except FileNotFoundError:
            pass
            
        return bboxes, class_labels
    
    def write_yolo_label(self, label_path, bboxes, class_labels):
        """Write YOLO format label file"""
        with open(label_path, 'w') as f:
            for bbox, class_id in zip(bboxes, class_labels):
                if all(0 <= coord <= 1 for coord in bbox):  # Ensure normalized coordinates
                    f.write(f"{int(class_id)} {bbox[0]:.6f} {bbox[1]:.6f} {bbox[2]:.6f} {bbox[3]:.6f}\n")
    
    def augment_image_and_labels(self, image_path, label_path):
        """Apply augmentation to image and corresponding labels"""
        # Read image
        image = cv2.imread(str(image_path))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Read labels
        bboxes, class_labels = self.read_yolo_label(label_path, image.shape[1], image.shape[0])
        
        if not bboxes:  # Skip if no bounding boxes
            return None, None, None
        
        # Apply augmentation
        try:
            augmented = self.transform(image=image, bboxes=bboxes, class_labels=class_labels)
            return augmented['image'], augmented['bboxes'], augmented['class_labels']
        except Exception as e:
            print(f"Augmentation failed for {image_path}: {e}")
            return None, None, None

def augment_yolo_dataset(input_dataset="combined_dataset", 
                        output_dataset="combined_with_augmentation",
                        augment_per_image=3,
                        augmentation_strength='medium',
                        splits_to_augment=['train']):
    """
    Main function to augment YOLO dataset
    
    Args:
        input_dataset: Path to input dataset
        output_dataset: Path to output augmented dataset (will contain originals + augmented)
        augment_per_image: Number of augmented versions per original image
        augmentation_strength: 'light', 'medium', or 'heavy'
        splits_to_augment: Which splits to augment (usually just 'train')
    """
    
    input_path = Path(input_dataset)
    output_path = Path(output_dataset)
    
    # Initialize augmenter
    augmenter = YOLOAugmentation(augmentation_strength)
    
    # Create output directory structure
    splits = ['train', 'valid', 'test']
    for split in splits:
        (output_path / split / 'images').mkdir(parents=True, exist_ok=True)
        (output_path / split / 'labels').mkdir(parents=True, exist_ok=True)
    
    # Track all files
    augmentation_log = []
    stats = {split: {'original': 0, 'augmented': 0} for split in splits}
    
    for split in splits:
        print(f"\n📁 Processing {split} split...")
        
        input_images_dir = input_path / split / 'images'
        input_labels_dir = input_path / split / 'labels'
        
        output_images_dir = output_path / split / 'images'
        output_labels_dir = output_path / split / 'labels'
        
        image_files = list(input_images_dir.glob('*.*'))
        
        for img_path in image_files:
            if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png']:
                label_path = input_labels_dir / f"{img_path.stem}.txt"
                
                # ✅ Copy original files to new dataset
                shutil.copy2(img_path, output_images_dir / img_path.name)
                if label_path.exists():
                    shutil.copy2(label_path, output_labels_dir / f"{img_path.stem}.txt")
                    stats[split]['original'] += 1
                
                augmentation_log.append({
                    'original_image': img_path.name,
                    'new_image': img_path.name,
                    'split': split,
                    'type': 'original',
                    'augmentation_id': 'original'
                })
                
                # ✅ Apply augmentation for specified splits
                if split in splits_to_augment and label_path.exists():
                    successful_augmentations = 0
                    
                    for aug_idx in range(augment_per_image):
                        aug_image, aug_bboxes, aug_classes = augmenter.augment_image_and_labels(
                            img_path, label_path
                        )
                        
                        if aug_image is not None and aug_bboxes:
                            # Generate unique ID for augmented file
                            aug_id = str(uuid.uuid4())
                            aug_image_name = f"{aug_id}{img_path.suffix}"
                            aug_label_name = f"{aug_id}.txt"
                            
                            # Save augmented image
                            aug_image_bgr = cv2.cvtColor(aug_image, cv2.COLOR_RGB2BGR)
                            cv2.imwrite(str(output_images_dir / aug_image_name), aug_image_bgr)
                            
                            # Save augmented labels
                            augmenter.write_yolo_label(output_labels_dir / aug_label_name, aug_bboxes, aug_classes)
                            
                            successful_augmentations += 1
                            stats[split]['augmented'] += 1
                            
                            augmentation_log.append({
                                'original_image': img_path.name,
                                'new_image': aug_image_name,
                                'split': split,
                                'type': 'augmented',
                                'augmentation_id': f"aug_{aug_idx+1}"
                            })
                    
                    print(f"  ✅ {img_path.name}: {successful_augmentations}/{augment_per_image} augmentations")
    
    # Save augmentation log
    log_df = pd.DataFrame(augmentation_log)
    log_df.to_csv(output_path / 'augmentation_log.csv', index=False)
    
    # Print summary statistics
    print("\n" + "="*50)
    print("🎉 AUGMENTATION COMPLETE!")
    print("="*50)
    
    total_original = 0
    total_augmented = 0
    
    for split in splits:
        orig = stats[split]['original']
        aug = stats[split]['augmented']
        total_original += orig
        total_augmented += aug
        
        if split in splits_to_augment:
            print(f"📊 {split.upper()}:")
            print(f"   Original images: {orig}")
            print(f"   Augmented images: {aug}")
            print(f"   Total: {orig + aug} images")
            print(f"   Increase: {((orig + aug) / orig - 1) * 100:.1f}%")
        else:
            print(f"📊 {split.upper()}: {orig} images (no augmentation)")
    
    print(f"\n📈 OVERALL STATISTICS:")
    print(f"   Total original images: {total_original}")
    print(f"   Total augmented images: {total_augmented}")
    print(f"   Final dataset size: {total_original + total_augmented} images")
    print(f"   Overall increase: {((total_original + total_augmented) / total_original - 1) * 100:.1f}%")
    
    print(f"\n💾 Output dataset: {output_path}")
    print(f"📋 Augmentation log: {output_path / 'augmentation_log.csv'}")
    
    return output_path

# Usage example:
if __name__ == "__main__":
    # Run augmentation
    augmented_dataset_path = augment_yolo_dataset(
        input_dataset="rebalanced_dataset",
        output_dataset="augmentation_dataset",
        augment_per_image=3,  # Create 3 augmented versions per training image
        augmentation_strength='medium',  # 'light', 'medium', or 'heavy'
        splits_to_augment=['train']  # Only augment training split
    )

C:\Users\plebj\AppData\Local\Temp\ipykernel_9280\1644128927.py:32: UserWarning: Argument(s) 'quality_lower, quality_upper' are not valid for transform ImageCompression
  A.ImageCompression(quality_lower=85, quality_upper=100, p=0.2),



📁 Processing train split...
  ✅ 00032c10-34d5-4a49-9105-9a28fe8415c5.jpg: 3/3 augmentations
  ✅ 005068e4-c186-48fe-b572-09c8ed37e5a0.jpg: 3/3 augmentations
  ✅ 0069c0f3-423f-4aef-8884-f4ef479136cc.jpg: 3/3 augmentations
  ✅ 00b0afcb-bead-42a0-8de4-fe5ad9cb1491.jpg: 3/3 augmentations
  ✅ 00f6bf1c-cb27-4c26-95fa-14d9905e5ab7.jpg: 3/3 augmentations
  ✅ 0107e8a6-47f8-49a6-98b1-e52c02a5a4c9.jpg: 3/3 augmentations
  ✅ 012fce0e-b602-423f-9974-751e8f219e0e.jpg: 3/3 augmentations
  ✅ 01462246-5c3f-455a-8808-eb27522ecc63.jpg: 3/3 augmentations
  ✅ 01768130-82b4-45ea-8110-701c35be0509.jpg: 3/3 augmentations
  ✅ 01a00fc2-ef5b-4e14-8068-c9125c18bce3.jpg: 3/3 augmentations
  ✅ 01beea7b-f4bd-4b5e-bace-eea2afeb2374.jpg: 3/3 augmentations
  ✅ 01e89d9d-445a-4e83-9f00-d701ad345377.jpg: 3/3 augmentations
  ✅ 021363ce-1102-4482-b600-82571eda2782.jpg: 3/3 augmentations
  ✅ 023e9b72-b5c4-489d-92c1-cafc0450249c.jpg: 3/3 augmentations
  ✅ 02958ca4-2f8e-4943-a5f3-90403ce1310e.jpg: 3/3 augmentations
  ✅ 02cb735

: 